# SDS 00 - PySpark Core Reference
Reusable exam-safe Spark patterns. Keep large data distributed; collect only small final results.

In [ ]:
# If `spark` is not pre-created:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName('SDS-reference').getOrCreate()
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Stable deterministic entity sampling

In [ ]:
def deterministic_sample(df, id_col, percent):
    return df.filter(F.pmod(F.xxhash64(F.col(id_col)), F.lit(100)) < F.lit(int(percent)))
# sample_1pct = deterministic_sample(events,'tracker_id',1)

## Multiple deterministic Spark hashes using salts

In [ ]:
def salted_hash(col_name, seed, modulus=None):
    h=F.xxhash64(F.concat(F.lit(str(seed)+'|'),F.col(col_name).cast('string')))
    return F.pmod(h,F.lit(int(modulus))) if modulus is not None else h

## Cache/materialize helper

In [ ]:
def cache_now(df):
    df=df.cache(); df.count(); return df

## Graph edge cleanup

In [ ]:
def clean_directed_edges(df, src='src', dst='dst'):
    return (df.select(F.col(src).cast('string').alias('src'),F.col(dst).cast('string').alias('dst'))
            .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
            .filter(F.col('src')!=F.col('dst')).distinct())

def canonical_undirected(edges):
    return (edges.select(F.least('src','dst').alias('u'),F.greatest('src','dst').alias('v'))
            .filter(F.col('u')!=F.col('v')).distinct())

## Iterative algorithm rule of thumb
For PageRank/HITS/SimRank-style algorithms: cache current state, compute next state with joins/aggregations, calculate a small scalar residual, unpersist old state, stop at tolerance or max_iter.

## Safe `collect()` rule
Good: top 25 results, a few scalars, a small query vector, a small list of community members after the heavy work is done. Bad: the full corpus or full large graph as the main computation.

## Exam answer scaffold
1. Recognize. 2. Formula. 3. Explicit parameters. 4. Distributed implementation. 5. Numerical verification. 6. Complexity/limitations. 7. Sanity-check output.